In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BigQuery AI/ML Integration Demo

<table align="left">

  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ai_ml_integration_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ai_ml_integration_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/github-logo.png" width="32" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/ai_ml_integration_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32">
      Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ai_ml_integration_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35">
      Open in BQ Studio
    </a>
  </td>
</table>

## Install Dependencies

In [2]:
import subprocess, sys

# Install SDK from local source (not yet published to PyPI).
# Falls back silently if packages are already available.
rc = subprocess.call(
    [sys.executable, "-m", "pip", "install", "-q",
     "--break-system-packages",
     "google-cloud-bigquery", "nest-asyncio",
     "-e", "../.."],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Verify the SDK is importable regardless of pip outcome
import bigquery_agent_analytics
print(f"bigquery_agent_analytics loaded (version: {getattr(bigquery_agent_analytics, '__version__', 'dev')})")

bigquery_agent_analytics loaded (version: dev)


## Authenticate & Configure

In [3]:
import os

# Colab authentication
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab — using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "test-project-0728-467323")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")
MODEL_NAME = os.environ.get("MODEL_NAME", "gemini-2.5-flash")
LOCATION = "US"
# AI.GENERATE requires a Vertex AI connection in BigQuery.
CONNECTION_ID = os.environ.get("BQ_CONNECTION_ID", "us.analytics-conn")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# Enable async in Jupyter
import nest_asyncio
nest_asyncio.apply()

print(f"Project      : {PROJECT_ID}")
print(f"Dataset      : {DATASET_ID}")
print(f"Table        : {TABLE_ID}")
print(f"Model        : {MODEL_NAME}")
print(f"Connection   : {CONNECTION_ID}")

Not running in Colab — using default credentials.
Project      : test-project-0728-467323
Dataset      : agent_analytics
Table        : agent_events
Model        : gemini-2.5-flash
Connection   : us.analytics-conn


## 1. LLM-Powered Evaluation (AI.GENERATE)

The `BigQueryAIClient` wraps BigQuery's **AI.GENERATE** scalar function for
text generation and trace analysis. AI.GENERATE is the recommended path:
it runs Gemini directly inside BigQuery with no model creation needed.

| Path | SQL Function | Model Required? | When to use |
|---|---|---|---|
| **Default** | `AI.GENERATE` | No (endpoint only) | Recommended for all new workloads |
| Legacy | `ML.GENERATE_TEXT` | Yes (BQ ML model) | Existing pipelines with pre-created models |

Key methods:
- `generate_text(prompt)` — raw text generation via AI.GENERATE
- `analyze_trace(trace_text, analysis_prompt)` — structured trace analysis (returns JSON)
- `generate_embeddings(texts)` — embedding generation (covered in Section 2)

In [4]:
import asyncio
import logging
from bigquery_agent_analytics.ai_ml_integration import BigQueryAIClient

# Suppress internal SDK warnings in notebook output — we handle errors below.
logging.getLogger("bigquery_agent_analytics").setLevel(logging.CRITICAL)

# --- Create the AI client (default: AI.GENERATE with gemini-2.5-flash) ---
ai_client = BigQueryAIClient(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    endpoint=MODEL_NAME,  # AI.GENERATE endpoint
    connection_id=CONNECTION_ID,  # Required for AI.GENERATE
    location=LOCATION,
)
print(f"BigQueryAIClient ready (endpoint: {ai_client.endpoint})")

# --- 1a. Generate text with AI.GENERATE ---
# Note: AI.GENERATE requires BigQuery Enterprise edition.  If your project
# does not have it, the call returns an empty result.  Other AI Operator
# functions (AI.EMBED, AI.DETECT_ANOMALIES, AI.FORECAST) do not require
# Enterprise and are demonstrated in later cells.
prompt = (
    "Summarize the key quality dimensions for evaluating an AI agent "
    "that helps users plan travel. List 3 dimensions in one sentence each."
)
result = asyncio.get_event_loop().run_until_complete(
    ai_client.generate_text(prompt, temperature=0.3, max_tokens=512)
)
print("\n[AI.GENERATE — Text Generation]")
print(f"Prompt : {prompt[:80]}...")
if result:
    print(f"Result : {result[:500]}")
else:
    print("Result : (empty — AI.GENERATE requires BigQuery Enterprise edition)")

# --- 1b. Analyze a trace with AI.GENERATE ---
sample_trace = """USER_MESSAGE_RECEIVED: Plan a trip from SF to Tokyo
TOOL_STARTING: search_flights
TOOL_COMPLETED: search_flights (3 results)
TOOL_STARTING: search_hotels
TOOL_COMPLETED: search_hotels (5 results)
TOOL_STARTING: get_weather_forecast
TOOL_COMPLETED: get_weather_forecast (sunny, 25C)
AGENT_COMPLETED: Here is your trip plan..."""

analysis = asyncio.get_event_loop().run_until_complete(
    ai_client.analyze_trace(
        trace_text=sample_trace,
        analysis_prompt=(
            "Analyze this agent trace for quality. Score task_completion, "
            "efficiency, and tool_usage each from 0 to 1."
        ),
    )
)
print("\n[AI.GENERATE — Trace Analysis]")
if analysis.get("raw_analysis"):
    for key, value in analysis.items():
        print(f"  {key}: {value}")
else:
    print("  (empty — AI.GENERATE requires BigQuery Enterprise edition)")

BigQueryAIClient ready (endpoint: gemini-2.5-flash)



[AI.GENERATE — Text Generation]
Prompt : Summarize the key quality dimensions for evaluating an AI agent that helps users...
Result : (empty — AI.GENERATE requires BigQuery Enterprise edition)



[AI.GENERATE — Trace Analysis]
  (empty — AI.GENERATE requires BigQuery Enterprise edition)


## 2. Semantic Search (AI.EMBED)

The `EmbeddingSearchClient` generates vector embeddings for agent traces
and performs semantic similarity search. Embeddings are stored in BigQuery
and searched using `ML.DISTANCE` (cosine similarity).

| Path | SQL Function | Model Required? | When to use |
|---|---|---|---|
| **Default** | `AI.EMBED` | No (endpoint only) | Recommended — scalar, no model creation |
| Legacy | `ML.GENERATE_EMBEDDING` | Yes (BQ ML model) | Existing pipelines with pre-created models |

The embedding generation uses `AI.EMBED` with `text-embedding-005` by
default. Pass `embedding_model="project.dataset.model"` to fall back
to the legacy `ML.GENERATE_EMBEDDING` path.

Key methods:
- `build_embeddings_index(since_days)` — index traces using AI.EMBED
- `search(query_embedding, top_k)` — vector similarity search via ML.DISTANCE

In [5]:
from bigquery_agent_analytics.ai_ml_integration import (
    EmbeddingSearchClient,
    EmbeddingResult,
)

# --- Create the embedding search client (default: AI.EMBED) ---
search_client = EmbeddingSearchClient(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    embeddings_table="trace_embeddings",
    source_table=TABLE_ID,
    embedding_endpoint="text-embedding-005",  # AI.EMBED endpoint
    # embedding_model="project.dataset.model",  # Uncomment for legacy ML.GENERATE_EMBEDDING
)
print(f"EmbeddingSearchClient ready (endpoint: {search_client.embedding_endpoint})")

# --- 2a. Generate embeddings using BigQueryAIClient.generate_embeddings ---
sample_texts = [
    "User asked about flights from SF to Tokyo",
    "Agent searched for hotels in Paris",
    "Weather forecast requested for New York",
]

embeddings = asyncio.get_event_loop().run_until_complete(
    ai_client.generate_embeddings(texts=sample_texts)
)
print(f"\n[AI.EMBED — Embedding Generation]")
print(f"Generated {len(embeddings)} embeddings:")
for emb in embeddings:
    dim = len(emb.embedding) if emb.embedding else 0
    preview = emb.embedding[:5] if emb.embedding else []
    print(f"  Text: {emb.text[:50]}")
    print(f"    Dimensions: {dim}, First 5 values: {preview}")

# --- 2b. Build embeddings index (indexes source traces) ---
# Note: build_embeddings_index creates a trace_embeddings table.
# If the table already exists, this is a no-op update.
try:
    success = asyncio.get_event_loop().run_until_complete(
        search_client.build_embeddings_index(since_days=7)
    )
    print(f"\n[AI.EMBED — Index Build]")
    print(f"Index build successful: {success}")
except Exception as exc:
    print(f"\n[AI.EMBED — Index Build]")
    print(f"  Skipped (table does not exist yet — run with more data to populate)")

# --- 2c. Search for similar sessions ---
try:
    if embeddings and embeddings[0].embedding:
        query_emb = embeddings[0].embedding
    else:
        query_emb = [0.01] * 768  # dummy 768-dim vector

    results = asyncio.get_event_loop().run_until_complete(
        search_client.search(
            query_embedding=query_emb,
            top_k=5,
            since_days=7,
        )
    )
    print(f"\n[ML.DISTANCE — Semantic Search]")
    print(f"Found {len(results)} similar sessions:")
    for r in results:
        print(f"  session={r['session_id']}  similarity={r['similarity']:.4f}")
        print(f"    content: {str(r['content'])[:80]}")
    if not results:
        print("  No results (embeddings index may be empty).")
except Exception:
    print(f"\n[ML.DISTANCE — Semantic Search]")
    print(f"  Skipped (trace_embeddings table not found — build index first)")

EmbeddingSearchClient ready (endpoint: text-embedding-005)



[AI.EMBED — Embedding Generation]
Generated 3 embeddings:
  Text: Weather forecast requested for New York
    Dimensions: 768, First 5 values: [-0.11003334820270538, 0.02058611623942852, 0.008281387388706207, -0.021092066541314125, -0.02532135508954525]
  Text: Agent searched for hotels in Paris
    Dimensions: 768, First 5 values: [-0.052303072065114975, 0.018068915233016014, -0.021042045205831528, 0.025477655231952667, 0.013758520595729351]
  Text: User asked about flights from SF to Tokyo
    Dimensions: 768, First 5 values: [0.02256159856915474, 0.0010262327268719673, -0.037858642637729645, 0.00012101876200176775, -0.003794343676418066]



[AI.EMBED — Index Build]
Index build successful: True



[ML.DISTANCE — Semantic Search]
Found 0 similar sessions:
  No results (embeddings index may be empty).


## 3. Anomaly Detection (AI.DETECT_ANOMALIES)

The `AnomalyDetector` identifies unusual latency patterns in agent traces
using time-series anomaly detection.

| Path | SQL Function | Model Required? | When to use |
|---|---|---|---|
| **Default** | `AI.DETECT_ANOMALIES` | No (built-in TimesFM) | Recommended — no model training needed |
| Legacy | `ML.DETECT_ANOMALIES` | Yes (ARIMA_PLUS model) | Existing pipelines with pre-trained models |

The default path uses **AI.DETECT_ANOMALIES** with the built-in TimesFM
foundation model. Set `use_legacy_anomaly_model=True` to fall back to
the legacy `ML.DETECT_ANOMALIES` path, which requires a pre-trained
ARIMA_PLUS model (via `train_latency_model()`).

Key methods:
- `detect_latency_anomalies(since_hours, training_days)` — find latency spikes
- `train_latency_model(training_days)` — train ARIMA_PLUS (legacy path only)
- `detect_behavior_anomalies(since_hours)` — AUTOENCODER-based behavioral detection

In [6]:
from bigquery_agent_analytics.ai_ml_integration import (
    AnomalyDetector,
    Anomaly,
    AnomalyType,
)

# --- Create the anomaly detector (default: AI.DETECT_ANOMALIES) ---
detector = AnomalyDetector(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    use_legacy_anomaly_model=False,  # True for legacy ML.DETECT_ANOMALIES
)
print(f"AnomalyDetector ready (legacy={detector.use_legacy_anomaly_model})")

# --- No model training needed for AI.DETECT_ANOMALIES ---
try:
    train_result = asyncio.get_event_loop().run_until_complete(
        detector.train_latency_model(training_days=30)
    )
    print(f"\n[Model Training]")
    print(f"train_latency_model() returned: {train_result}")
    print("(AI.DETECT_ANOMALIES uses built-in TimesFM — no training needed.)")
except Exception as exc:
    print(f"Model training check failed: {exc}")

# --- Detect latency anomalies ---
try:
    anomalies = asyncio.get_event_loop().run_until_complete(
        detector.detect_latency_anomalies(
            since_hours=24,
            training_days=30,
        )
    )
    print(f"\n[AI.DETECT_ANOMALIES — Latency Anomalies]")
    print(f"Detected {len(anomalies)} anomalies in the last 24 hours:")
    for a in anomalies:
        print(f"  [{a.anomaly_type.value}] severity={a.severity:.2f}")
        print(f"    {a.description}")
        print(f"    timestamp : {a.timestamp}")
        print(f"    details   : {a.details}")
    if not anomalies:
        print("  No anomalies found (this is normal for healthy systems).")
except Exception as exc:
    print(f"Latency anomaly detection failed: {exc}")
    print("(This is expected if the events table has insufficient time-series data.)")

AnomalyDetector ready (legacy=False)

[Model Training]
train_latency_model() returned: True
(AI.DETECT_ANOMALIES uses built-in TimesFM — no training needed.)



[AI.DETECT_ANOMALIES — Latency Anomalies]
Detected 0 anomalies in the last 24 hours:
  No anomalies found (this is normal for healthy systems).


## 4. Latency Forecasting (AI.FORECAST)

The `AnomalyDetector` also provides latency forecasting to predict future
latency trends. This is useful for capacity planning and proactive alerting.

| Path | SQL Function | Model Required? | When to use |
|---|---|---|---|
| **Default** | `AI.FORECAST` | No (built-in TimesFM) | Recommended — no model training needed |
| Legacy | `ML.FORECAST` | Yes (ARIMA_PLUS model) | Existing pipelines with pre-trained models |

Each forecast point is a `LatencyForecast` dataclass with:
- `timestamp` — forecasted time point
- `forecast_value` — predicted latency in ms
- `lower_bound` / `upper_bound` — prediction interval
- `status` — `ai_forecast_status` (empty string = success)

In [7]:
from bigquery_agent_analytics.ai_ml_integration import LatencyForecast

# --- Forecast latency for the next 12 hours ---
try:
    forecasts = asyncio.get_event_loop().run_until_complete(
        detector.forecast_latency(
            horizon_hours=12,
            training_days=30,
            confidence_level=0.95,
        )
    )
    print(f"[AI.FORECAST — Latency Forecasting]")
    print(f"Forecasted {len(forecasts)} hourly data points:")

    # Filter to successful forecasts only
    ok_forecasts = [f for f in forecasts if not f.status]
    err_forecasts = [f for f in forecasts if f.status]

    print(f"  Successful: {len(ok_forecasts)}, Skipped: {len(err_forecasts)}")

    for f in ok_forecasts[:5]:  # Show first 5
        print(
            f"  {f.timestamp}  "
            f"forecast={f.forecast_value:.1f}ms  "
            f"[{f.lower_bound:.1f}, {f.upper_bound:.1f}]"
        )
    if len(ok_forecasts) > 5:
        print(f"  ... ({len(ok_forecasts) - 5} more points)")

    if err_forecasts:
        print(f"\n  Skipped {len(err_forecasts)} point(s) — insufficient time-series history:")
        for f in err_forecasts[:3]:
            print(f"    {f.timestamp}  reason: {f.status}")

    if not ok_forecasts:
        print("\n  (AI.FORECAST needs longer time-series history for meaningful predictions.)")
except Exception as exc:
    print(f"Latency forecasting failed: {exc}")
    print("(This is expected if the events table has insufficient time-series data.)")

[AI.FORECAST — Latency Forecasting]
Forecasted 1 hourly data points:
  Successful: 0, Skipped: 1

  Skipped 1 point(s) — insufficient time-series history:
    2026-04-07 06:42:30.495685+00:00  reason: The time series data is too short.

  (AI.FORECAST needs longer time-series history for meaningful predictions.)


## 5. Batch Evaluation

The `BatchEvaluator` leverages BigQuery's high-throughput **AI.GENERATE**
to evaluate many sessions in a single query. It uses `output_schema` for
typed structured results (task_completion, efficiency, tool_usage).

This is much more efficient than evaluating sessions one-by-one because
BigQuery parallelizes the LLM calls across its distributed infrastructure.

Key methods:
- `evaluate_recent_sessions(days, limit)` — batch-evaluate sessions
- `store_evaluation_results(results, table_name)` — persist results to BigQuery

In [8]:
from bigquery_agent_analytics.ai_ml_integration import (
    BatchEvaluator,
    BatchEvaluationResult,
)

# --- Create the batch evaluator (default: AI.GENERATE) ---
batch_evaluator = BatchEvaluator(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    endpoint=MODEL_NAME,  # AI.GENERATE endpoint
    connection_id=CONNECTION_ID,  # Required for AI.GENERATE
    # eval_model="project.dataset.model",  # Uncomment for legacy ML.GENERATE_TEXT
)
print(f"BatchEvaluator ready (endpoint: {batch_evaluator.endpoint})")

# --- Evaluate recent sessions in batch ---
# Note: Like generate_text above, batch evaluation uses AI.GENERATE and
# requires BigQuery Enterprise edition.
eval_results = asyncio.get_event_loop().run_until_complete(
    batch_evaluator.evaluate_recent_sessions(
        days=7,
        limit=10,
    )
)
print(f"\n[AI.GENERATE — Batch Evaluation]")
if eval_results:
    print(f"Evaluated {len(eval_results)} sessions:")
    for r in eval_results:
        status = "ERROR" if r.error else "OK"
        print(
            f"  {r.session_id}: "
            f"task_completion={r.task_completion:.2f}  "
            f"efficiency={r.efficiency:.2f}  "
            f"tool_usage={r.tool_usage:.2f}  "
            f"[{status}]"
        )
else:
    print("  No results (AI.GENERATE requires BigQuery Enterprise edition,"
          " or no sessions found in the specified time range)")

BatchEvaluator ready (endpoint: gemini-2.5-flash)



[AI.GENERATE — Batch Evaluation]
  No results (AI.GENERATE requires BigQuery Enterprise edition, or no sessions found in the specified time range)


## Summary

This notebook demonstrated the **BigQuery AI/ML Integration** module
(Section 13) of the BigQuery Agent Analytics SDK.

| # | Feature | Class | Default SQL | Legacy Fallback |
|---|---|---|---|---|
| 1 | LLM-Powered Evaluation | `BigQueryAIClient` | `AI.GENERATE` | `ML.GENERATE_TEXT` |
| 2 | Semantic Search | `EmbeddingSearchClient` | `AI.EMBED` + `ML.DISTANCE` | `ML.GENERATE_EMBEDDING` |
| 3 | Anomaly Detection | `AnomalyDetector` | `AI.DETECT_ANOMALIES` | `ML.DETECT_ANOMALIES` (ARIMA_PLUS) |
| 4 | Latency Forecasting | `AnomalyDetector` | `AI.FORECAST` | `ML.FORECAST` (ARIMA_PLUS) |
| 5 | Batch Evaluation | `BatchEvaluator` | `AI.GENERATE` (with `output_schema`) | `ML.GENERATE_TEXT` |

### Key Takeaways

- **AI Operators are the default** — `AI.GENERATE`, `AI.EMBED`, `AI.DETECT_ANOMALIES`,
  and `AI.FORECAST` require no model creation or training.
- **Legacy fallbacks available** — Pass a pre-created BQ ML model reference
  (e.g., `embedding_model="project.dataset.model"`) or set
  `use_legacy_anomaly_model=True` to use the legacy `ML.*` paths.
- **Batch evaluation is efficient** — `BatchEvaluator` runs LLM calls in
  parallel using BigQuery's distributed infrastructure.
- **LatencyForecast.status** — Filter `[f for f in forecasts if not f.status]`
  to get only successful forecast points.
- **Behavioral anomaly detection** — Uses `ML.DETECT_ANOMALIES` with
  AUTOENCODER (no AI Operator equivalent for non-time-series data).

In [9]:
print("Notebook complete!")

Notebook complete!
